# 🎤 YingMusic-SVC — Google Colab (ฉบับแก้ไขสมบูรณ์)

เปลี่ยนเสียงร้องในเพลงให้เป็นเสียงของคนอื่นแบบ Zero-Shot (ใช้แค่ตัวอย่างเสียง 10–25 วินาที)

1. ตั้งค่า `Runtime → Change runtime type → T4 GPU`
2. กดปุ่ม ▶ ที่ Cell ด้านล่าง
3. รอติดตั้งและดาวน์โหลดโมเดล (เฉพาะครั้งแรก ~5–8 นาที) → จากนั้นจะมีปุ่ม **Choose Files** ให้อัปโหลดเพลงต้นฉบับ และเสียงเป้าหมาย (ถ้าเลือก `upload`)
4. รอประมวลผลเสร็จ → สามารถกดฟังผลลัพธ์ได้ใน Notebook และเบราว์เซอร์จะดาวน์โหลดไฟล์ให้อัตโนมัติ

หากต้องการแปลงเพลงถัดไป สามารถกดปุ่ม ▶ ซ้ำได้เลยโดยไม่ต้องติดตั้งใหม่

In [ ]:
#@title ▶ แปลงเสียงเพลง (YingMusic-SVC)
#@markdown **เสียงเป้าหมาย** — `upload` = อัปโหลดเสียงร้อง/พูดที่สะอาด ไม่มีดนตรี ยาว 10–25 วินาที
TIMBRE = "upload" #@param ["upload", "prompts/tp.wav", "prompts/syz.wav"]
#@markdown ติ๊กถูกถ้าเพลงมีดนตรีประกอบ (ระบบจะแยกเสียงร้องด้วย BS-RoFormer ให้ก่อน) — เอาติ๊กออกถ้าเป็นไฟล์เสียงร้องล้วน
HAS_ACCOMPANIMENT = True #@param {type:"boolean"}
#@markdown ติ๊กถูกถ้าอยากเก็บเสียงคอรัส (Backing Vocal) ของต้นฉบับไว้ในดนตรีตอนมิกซ์กลับ
KEEP_BACKING_VOCAL = False #@param {type:"boolean"}
#@markdown คุณภาพ vs ความเร็ว: 100 = ค่าแนะนำสูงสุด, 30–50 = เร็วขึ้น
DIFFUSION_STEPS = 100 #@param {type:"slider", min:10, max:200, step:10}
#@markdown ---
#@markdown **ตั้งค่าขั้นสูง (ปกติไม่ต้องแก้ไข)**
REPO_URL = "https://github.com/GiantAILab/YingMusic-SVC" #@param {type:"string"}
BRANCH = "" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}

import os
import shlex
import shutil
import subprocess
import time
import glob
from google.colab import files
from IPython.display import Audio, display

def sh(cmd, cwd=None):
    """รันคำสั่ง Shell แสดงผลแบบสด และหยุดทันทีเมื่อเกิดข้อผิดพลาด"""
    p = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise SystemExit(f"❌ คำสั่งล้มเหลว (exit {p.returncode}): {cmd}")

q = shlex.quote
REPO_DIR = "/content/YingMusic-SVC"
READY = "/content/.ymsvc_ready"

if subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise SystemExit("❌ ไม่พบ GPU — กรุณาไปที่เมนู Runtime → Change runtime type → เลือก T4 GPU แล้วรันใหม่อีกครั้ง")

# ================= 1. ติดตั้งและเตรียมโมเดล (ทำครั้งแรกครั้งเดียว) =================
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"📥 กำลังโคลนโปรเจกต์จาก {REPO_URL} ...")
    branch_arg = f"-b {q(BRANCH)}" if BRANCH else ""
    sh(f"git clone --depth 1 {branch_arg} {q(REPO_URL)} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull -q --ff-only")

os.chdir(REPO_DIR)
sh('git log -1 --format="📌 Code version: %h %s"')

if not os.path.exists(READY):
    print("📦 กำลังติดตั้งไลบรารีที่จำเป็น (ใช้เวลาประมาณ 2-3 นาที)...")
    sh("apt-get -qq update && apt-get -qq install -y sox libsox-fmt-all ffmpeg > /dev/null")

    # ติดตั้งแพ็กเกจโดยไม่ทับ torch/torchaudio ของ Colab เพื่อป้องกันปัญหา CUDA mismatch
    pkgs = (
        '"librosa>=0.10.2,<1" "transformers>=4.46.3,<5" "huggingface-hub>=0.28.1,<1.0" '
        'munch einops "descript-audio-codec==1.0.0" "gradio>=5.23,<6" pydub soundfile pyyaml '
        'omegaconf ml_collections loralib wandb beartype "rotary_embedding_torch==0.3.5" '
        'matplotlib scipy tqdm audiomentations pedalboard'
    )
    sh(f"pip install -q {pkgs}")

    print("⬇️ กำลังดาวน์โหลดโมเดลจาก Hugging Face...")
    from huggingface_hub import hf_hub_download, snapshot_download
    token = HF_TOKEN.strip() or None

    os.makedirs("checkpoints", exist_ok=True)
    os.makedirs("accom_separation/ckpt/bs_roformer", exist_ok=True)

    # 1. โมเดลหลัก YingMusic-SVC และ BS-RoFormer
    hf_hub_download(
        repo_id="GiantAILab/YingMusic-SVC",
        filename="YingMusic-SVC-full.pt",
        local_dir="checkpoints",
        token=token
    )
    hf_hub_download(
        repo_id="GiantAILab/YingMusic-SVC",
        filename="bs_roformer.ckpt",
        local_dir="accom_separation/ckpt/bs_roformer",
        token=token
    )

    # 2. โมเดลเสริม RMVPE และ CAMPPlus (โหลดทั้งแบบ cache_dir และ local_dir เพื่อให้รองรับทุกการตั้งค่า)
    rmvpe_cached = hf_hub_download("lj1995/VoiceConversionWebUI", "rmvpe.pt", cache_dir="./checkpoints", token=token)
    camp_cached = hf_hub_download("funasr/campplus", "campplus_cn_common.bin", cache_dir="./checkpoints", token=token)
    if not os.path.exists("checkpoints/rmvpe.pt"):
        shutil.copy(rmvpe_cached, "checkpoints/rmvpe.pt")
    if not os.path.exists("checkpoints/campplus_cn_common.bin"):
        shutil.copy(camp_cached, "checkpoints/campplus_cn_common.bin")

    # 3. โมเดล BigVGAN และ Whisper-Small
    snapshot_download(
        repo_id="nvidia/bigvgan_v2_44khz_128band_512x",
        cache_dir="./checkpoints/hf_cache",
        allow_patterns=["config.json", "bigvgan_generator.pt"],
        token=token
    )
    snapshot_download(
        repo_id="nvidia/bigvgan_v2_44khz_128band_512x",
        local_dir="./nvidia/bigvgan_v2_44khz_128band_512x",
        allow_patterns=["config.json", "bigvgan_generator.pt"],
        token=token
    )
    snapshot_download(
        repo_id="openai/whisper-small",
        cache_dir="./checkpoints/hf_cache",
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        token=token
    )
    snapshot_download(
        repo_id="openai/whisper-small",
        local_dir="./openai/whisper-small",
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"],
        token=token
    )

    sh("python -c \"import torch, torchaudio, librosa, transformers, dac; print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())\"")
    open(READY, "w").close()
    print("✅ ติดตั้งระบบและดาวน์โหลดโมเดลเสร็จสมบูรณ์!")

SVC_CKPT = os.path.abspath("checkpoints/YingMusic-SVC-full.pt")
SEP_CKPT = os.path.abspath("accom_separation/ckpt/bs_roformer/bs_roformer.ckpt")
SEP_CFG = os.path.abspath("accom_separation/ckpt/bs_roformer/config_bd_roformer.yaml")

# ================= 2. อัปโหลดไฟล์เสียง =================
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
WORK = os.path.abspath(f"colab_runs/{RUN_ID}")
os.makedirs(WORK, exist_ok=True)

def upload_one(prompt_msg):
    print(f"\n{prompt_msg}")
    up = files.upload()
    if not up:
        raise SystemExit("❌ ยกเลิกการทำงานเนื่องจากไม่ได้อัปโหลดไฟล์")
    name = next(iter(up))
    safe_name = os.path.basename(name).replace(" ", "_")
    dst = os.path.join(WORK, "upload_" + safe_name)
    shutil.move(name, dst)
    return dst, os.path.splitext(safe_name)[0]

song_raw, song_title = upload_one("🎵 ขั้นตอนที่ 1: อัปโหลดเพลงต้นฉบับ (รองรับ mp3 / wav / flac / m4a):")
SONG_WAV = os.path.join(WORK, "song.wav")
sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(song_raw)} -ar 44100 {q(SONG_WAV)}")

if TIMBRE == "upload":
    timbre_raw, _ = upload_one("🗣️ ขั้นตอนที่ 2: อัปโหลดไฟล์เสียงเป้าหมาย (เสียงร้องหรือพูดที่ไม่มีดนตรีปน ความยาว 10–25 วินาที):")
    TIMBRE_WAV = os.path.join(WORK, "timbre.wav")
    sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(timbre_raw)} -ar 44100 -ac 1 {q(TIMBRE_WAV)}")
else:
    TIMBRE_WAV = os.path.abspath(TIMBRE)

# ================= 3. แยกเสียงร้องและดนตรีประกอบ =================
if HAS_ACCOMPANIMENT:
    print("\n🎚️ กำลังแยกเสียงร้องออกจากดนตรีประกอบด้วย BS-RoFormer...")
    sep_in = os.path.join(WORK, "sep_in")
    sep_out = os.path.join(WORK, "sep_out")
    os.makedirs(sep_in, exist_ok=True)
    shutil.copy(SONG_WAV, sep_in)
    sh(
        f"PYTHONWARNINGS=ignore python inference.py "
        f"--model_type bs_roformer "
        f"--config_path {q(SEP_CFG)} "
        f"--start_check_point {q(SEP_CKPT)} "
        f"--input_folder {q(sep_in)} "
        f"--store_dir {q(sep_out)} "
        f"--disable_detailed_pbar",
        cwd="accom_separation"
    )
    stems = os.path.join(sep_out, "song")
    SOURCE_VOCAL = os.path.join(stems, "vocals.wav")
    ACCOMPANY = os.path.join(stems, "instrumental.wav")

    if KEEP_BACKING_VOCAL:
        backing_file = os.path.join(stems, "backing_vocal.wav")
        if os.path.exists(backing_file):
            mixed_acc = os.path.join(stems, "instrumental_with_backing.wav")
            sh(
                f"ffmpeg -hide_banner -loglevel error -y "
                f"-i {q(ACCOMPANY)} -i {q(backing_file)} "
                f"-filter_complex amix=inputs=2:normalize=0 {q(mixed_acc)}"
            )
            ACCOMPANY = mixed_acc
else:
    SOURCE_VOCAL = SONG_WAV
    ACCOMPANY = ""

# ================= 4. แปลงเสียงร้อง (Zero-Shot SVC) =================
print("\n🎤 กำลังแปลงเสียงร้อง... (เพลงความยาว 3–4 นาทีบน T4 GPU จะใช้เวลาประมาณ 3–6 นาที)")
EXP = f"colab_{RUN_ID}"
accompany_arg = f"--accompany {q(ACCOMPANY)}" if ACCOMPANY else ""

sh(
    f"python -u my_inference.py "
    f"--source {q(SOURCE_VOCAL)} "
    f"--target {q(TIMBRE_WAV)} "
    f"--diffusion-steps {int(DIFFUSION_STEPS)} "
    f"--checkpoint {q(SVC_CKPT)} "
    f"--expname {q(EXP)} "
    f"--cuda 0 "
    f"--fp16 True "
    f"{accompany_arg} "
    f"--config configs/YingMusic-SVC.yml"
)

vc_files = sorted(glob.glob(f"outputs/{EXP}/*.wav"))
mix_files = sorted(glob.glob(f"outputs/{EXP}/accompany/*.wav"))

if not vc_files:
    raise SystemExit("❌ ไม่พบไฟล์ผลลัพธ์ กรุณาตรวจสอบข้อความแจ้งเตือนด้านบน")

# ================= 5. แสดงผลและดาวน์โหลดไฟล์ =================
results = [("เสียงร้องเดี่ยวที่แปลงแล้ว (Vocal Only)", vc_files[-1], f"{song_title}_vocal.wav")]
if mix_files:
    results.insert(0, ("เพลงเต็มที่มิกซ์กับดนตรีแล้ว (Full AI Cover)", mix_files[-1], f"{song_title}_cover.wav"))

print("\n🎉 ประมวลผลเสร็จเรียบร้อย!")
for label, wav_path, nice_name in results:
    final_wav = os.path.join(WORK, nice_name)
    shutil.copy(wav_path, final_wav)
    preview_mp3 = os.path.splitext(final_wav)[0] + ".mp3"
    sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(final_wav)} -b:a 192k {q(preview_mp3)}")
    print(f"\n✅ {label}:")
    display(Audio(filename=preview_mp3))
    files.download(final_wav)